# Fit Production Atlases and Models

Fits the core EmbAlign models (GP Atlas, Slice Atlas, and Diagnostic Oracle) on the complete dataset. This notebook exports the final trained spatial and diagnostic artifacts required to run the pipeline on unannotatedembryos. We also fit the empirical growth curve used in the alignment dashboard.

In [11]:
# Standard Imports
import pandas as pd
import numpy as np
import joblib
import os
from tqdm import tqdm

# Import EmbAlign Modules
from aligner.config import PipelineConfig
from aligner.atlas import AtlasFactory 
from aligner.matcher import SinkhornMatcher, HungarianMatcher
from aligner.transformer import RigidTransformer
from aligner.engine import ModularAlignmentEngine
from aligner.models import EmbryoFrame
from aligner.oracle import DiagnosticLayer
from aligner.plot_utils import *

## Fit Spatiotemporal and Slice Atlases

In [ ]:
# Read in labled data
PTS_CSV = "data/final_registered_all_pts.csv"
pts = pd.read_csv(PTS_CSV)
pts.head()

,embryo_id,time_idx,cell_name,x_um,y_um,z_um,valid,n_cells_frame,source_file,canonical_time,x_aligned,y_aligned,z_aligned
0,191108,13,ABar,45.18,23.22,16.548,1,6,t013-nuclei,23.0,1.375155,1.168478,1.185836
1,191108,13,ABpl,28.44,17.64,20.370,1,6,t013-nuclei,23.0,2.284725,0.802801,0.964768
2,191108,13,EMS,30.42,35.28,17.598,1,6,t013-nuclei,23.0,2.231156,1.792119,1.100340
3,191108,13,P2,14.94,24.66,18.480,1,6,t013-nuclei,23.0,3.058124,1.153517,1.049814
4,191108,13,ABpr,29.07,17.28,15.960,1,6,t013-nuclei,23.0,2.252809,0.789615,1.211125


In [15]:
# Initialize and configure pipeline
config = PipelineConfig.v3_0_production()
factory = AtlasFactory(pts, config)

# Fit spatiotemporal and slice atlases
all_embryos = pts['embryo_id'].unique()
master_spatial_atlas, master_slice_db = factory.build(all_embryos)

# Extract the life history dataframe
master_life_history = factory.life_history

In [16]:
# Cache
output_dir = "production_models"
os.makedirs(output_dir, exist_ok=True)

gp_atlas_path = os.path.join(output_dir, 'master_gp_atlas.pkl')
slice_db_path = os.path.join(output_dir, 'master_slice_db.pkl')
life_history_path = os.path.join(output_dir, 'master_life_history.pkl')

joblib.dump(master_spatial_atlas, gp_atlas_path)
joblib.dump(master_slice_db, slice_db_path)
joblib.dump(master_life_history, life_history_path)

['production_models/master_life_history.pkl']

## Train Production Oracle

In [7]:
# Read in cell level results from CV sweep.
CV_CELLS = "data/aligned_cell_results_cv.csv"
cv_diagnostics_df = pd.read_csv(CV_CELLS)
cv_diagnostics_df.head()

,frame_id,embryo_id,time_idx,cell_name,lineage,x_aligned,y_aligned,z_aligned,mah_dist,entropy,map_time,frame_is_generated,num_cells_in_frame,div_delta,is_correct,config_name,confidence_score
0,191108_tid13,191108,13,ABar,AB,1.221425,1.098740,1.131822,1.397235,6.479153e-04,23.933333,False,6,0.066194,True,EmbAlign_CV,0.986276
1,191108_tid13,191108,13,ABpl,AB,2.137628,0.748932,0.912473,1.023309,4.719417e-06,23.933333,False,6,0.070707,True,EmbAlign_CV,0.994617
2,191108_tid13,191108,13,EMS,EMS,2.050159,1.743488,0.969113,16.981287,1.431551e-10,23.933333,False,6,0.424242,True,EmbAlign_CV,0.982529
3,191108_tid13,191108,13,P2,P2,2.899162,1.132896,0.944369,0.656823,1.431551e-10,23.933333,False,6,0.140351,True,EmbAlign_CV,0.996797
4,191108_tid13,191108,13,ABpr,AB,2.113288,0.754881,1.159973,4.361789,5.896947e-03,23.933333,False,6,0.070707,True,EmbAlign_CV,0.977389


In [8]:
# Train oracle
oracle = DiagnosticLayer(training_data=cv_diagnostics_df)
print(oracle.get_performance_summary(cv_diagnostics_df))
print(oracle.get_feature_importance_df())

DiagnosticLayer: Model fitted on training data.
{'0': {'precision': 0.348145285935085, 'recall': 0.9203268641470889, 'f1-score': 0.5051864311746566, 'support': 4895.0}, '1': {'precision': 0.9949070204763895, 'recall': 0.9003202514742203, 'f1-score': 0.9452533235730194, 'support': 84621.0}, 'accuracy': 0.901414272308861, 'macro avg': {'precision': 0.6715261532057373, 'recall': 0.9103235578106545, 'f1-score': 0.725219877373838, 'support': 89516.0}, 'weighted avg': {'precision': 0.9595401733140978, 'recall': 0.901414272308861, 'f1-score': 0.9211891625482865, 'support': 89516.0}}
              feature  importance
1             entropy    0.467984
3           div_delta    0.209650
0            mah_dist    0.137681
2            map_time    0.096922
4  num_cells_in_frame    0.087763


In [9]:
# Cache
oracle_path = os.path.join(output_dir, 'production_oracle.pkl')
oracle.save_model(oracle_path)

Model saved to production_models/production_oracle.pkl


## Fit Empirical Growth Curve for Alignment Dashboard

In [17]:
growth_curve = build_empirical_growth_curve(pts, bin_size=1.0, output_path='production_models/empirical_growth_curve.csv')

Saved empirical growth curve to production_models/empirical_growth_curve.csv
